# MPPI on MuJoCo with GPU-parallel rollouts and mjviser visualization

This notebook walks through:
1. Installing `mpc-warp` and its dependencies
2. Instantiating a built-in task (inverted pendulum swingup)
3. Running MPPI with `WarpMPPISolver` (GPU via CUDA / CPU Warp kernels)
4. Live visualization in the browser with `mjviser` + `MppiPanel`

> **Google Colab GPU**: Runtime → Change runtime type → **T4 GPU** before running.
> The solver automatically picks `cuda:0` when a GPU is present.

## 1 · Install dependencies

One cell installs `mpc-warp` from GitHub (pulls in MuJoCo, mujoco-warp, mjviser, etc. from `pyproject.toml`) plus `matplotlib` for the optional cost plot at the end.

In [ ]:
!pip install -q uv

!uv pip install -q --reinstall \
    "git+https://github.com/Vittorio-Caggiano/mpc-warp.git@main" \
    matplotlib

# On Colab: Runtime → Restart session after this cell, then re-run from cell 2.
print("Install complete.")

## 2 · Verify GPU availability

In [ ]:
import warp as wp

wp.init()
print("Warp device:", wp.get_device().alias)

## 3 · Create a task and environment

A **Task** pairs a MuJoCo model with cost functions:
- `running_cost(data, ctrl)` — evaluated at every rollout step
- `terminal_cost(data)` — evaluated at the end of the horizon

`MujocoTaskEnv` wraps the task into a `reset / step` interface and holds the
live `mujoco.MjData` that both the solver and the viewer share.

In [ ]:
from mpc_warp.envs.mujoco_env import MujocoTaskEnv
from mpc_warp.tasks import Pendulum

task = Pendulum()  # inverted-pendulum swingup
env = MujocoTaskEnv(task)
env.reset()

print("nq:", task.mj_model.nq, "  nv:", task.mj_model.nv, "  nu:", task.mj_model.nu)
print("u_min:", task.u_min, "  u_max:", task.u_max)
print("trace site ids:", task.trace_site_ids)

## 4 · Create the MPPI solver

`WarpMPPISolver` allocates `num_samples` parallel MuJoCo worlds on the Warp device.
Each `solver.command(env.data)` call:
1. Uploads the current state to all worlds
2. Rolls out `horizon` steps in parallel
3. Weights samples by `exp(-cost / temperature)`
4. Returns the first action of the updated nominal trajectory

In [ ]:
import warp as wp

from mpc_warp.solvers.mppi_warp import WarpMPPIConfig, WarpMPPISolver

# NOTE: the cost loop reads back every sample to CPU each step (N×H transfers),
# so num_samples is the main performance knob regardless of GPU vs CPU.
# Keep it small until a vectorised cost kernel is available.
device = wp.get_device().alias
num_samples = 64 if device == "cpu" else 128

cfg = WarpMPPIConfig(
    horizon=16,
    num_samples=num_samples,
    noise_sigma=0.5,
    temperature=0.1,
)
solver = WarpMPPISolver(task, cfg, seed=0)
print(f"Solver device: {solver.device}  |  num_samples: {num_samples}")
print("Note: first command() call compiles Warp kernels — expect ~1 min on a cold Colab.")

## 5 · Launch the mjviser browser viewer

`viser.ViserServer` starts a WebSocket server (default port 8080).  
Open the printed URL in a browser to see the 3-D physics scene and the MPPI
diagnostic panels (cost history, ESS, action bars, nominal trajectory).

> **Colab**: the cell embeds the viewer inline via an `IFrame`.  
> **Re-running this cell** stops the previous server automatically before starting a new one.

In [ ]:
import mujoco
import viser
from mjviser import ViserMujocoScene

from mpc_warp.viz.mjviser_panel import MppiPanel

# Stop any server left over from a previous run of this cell.
_prev_server = globals().get("server")
if _prev_server is not None:
    _prev_server.stop()

server = viser.ViserServer()  # listens on :8080 by default

act_names = [mujoco.mj_id2name(task.mj_model, mujoco.mjtObj.mjOBJ_ACTUATOR, i) or f"a{i}" for i in range(task.mj_model.nu)]
panel = MppiPanel(server, task.mj_model.nu, act_names)
scene = ViserMujocoScene(server, task.mj_model, num_envs=1)

port = server.get_port()
print(f"Open in browser: http://localhost:{port}")

# Colab: embed inline
try:
    from google.colab.output import eval_js  # type: ignore
    from IPython.display import IFrame, display

    colab_url = eval_js(f"google.colab.kernel.proxyPort({port})")
    display(IFrame(src=colab_url, width="100%", height=520))
except ImportError:
    pass  # not on Colab — open the URL above manually

## 6 · Run the control loop

Each iteration:
1. `solver.command` → optimal action from MPPI
2. `env.step` → advance the physics
3. `scene.update_from_mjdata` → push new pose to browser
4. `panel.update` → refresh cost / ESS / action panels

In [ ]:
import time

NUM_STEPS = 500
REALTIME = True  # False → run as fast as possible
LOG_EVERY = 50

env.reset()
total_cost = 0.0
step_start = time.perf_counter()

for step_i in range(NUM_STEPS):
    u = solver.command(env.data)
    _, reward, *_ = env.step(u)
    total_cost -= reward

    scene.update_from_mjdata(env.data)
    panel.update(
        solver.last_cost,
        solver.cost_weights,
        u,
        cost_terms=solver.last_cost_terms,
        u_nominal=solver.u_nominal_snapshot,
    )

    if REALTIME:
        elapsed = time.perf_counter() - step_start
        target = (step_i + 1) * task.dt
        if target > elapsed:
            time.sleep(target - elapsed)

    if (step_i + 1) % LOG_EVERY == 0:
        print(f"step {step_i + 1:4d}  cost={solver.last_cost:.3f}  cumulative={total_cost:.2f}")

print(f"\nDone — {NUM_STEPS} steps, total cost = {total_cost:.2f}")

## 7 · Plot the cost history in-notebook

In [ ]:
import matplotlib.pyplot as plt

env.reset()
costs = []
for _ in range(NUM_STEPS):
    u = solver.command(env.data)
    env.step(u)
    costs.append(solver.last_cost)

plt.figure(figsize=(9, 3))
plt.plot(costs)
plt.xlabel("step")
plt.ylabel("running cost")
plt.title("MPPI running cost — Pendulum swingup")
plt.tight_layout()
plt.show()

## 8 · Tear down the server

In [ ]:
server.stop()

## Next steps

- **Swap the task**: replace `Pendulum()` with `CartPole()`, `Walker()`, `Crane()`, `HumanoidStandup()`, etc.
- **Custom task**: subclass `Task`, supply a MuJoCo XML model, and implement `running_cost` + `terminal_cost`.
- **More samples on GPU**: push `num_samples` to 512–2048; the Warp backend scales linearly.
- **Trajectory tracking**: wrap any base task with `TrajectoryTask` and call `task.advance()` after each `env.step`.